# Q3 training queue on Kaggle

Runs the remaining NRMS / popularity-aware NRMS runs from `tools/run_q3.sh` on Kaggle GPUs.

**Settings (right panel):** Accelerator = **GPU T4 x2**, Persistence = off, Internet = on (only needed if a pip fix is required).
Attach the private dataset containing `ire-team/` (uploaded from `q3_kaggle_bundle.zip`).

Run the first code cell interactively to check the smoke test passes, then **Save Version -> Save & Run All (Commit)**
so the queue keeps running after you close the browser. Download `q3_results.zip` from the version's **Output** tab.

In [ ]:
%%bash
# 1. Copy the bundle somewhere writable and check the environment.
set -e
SRC=$(dirname "$(dirname "$(find /kaggle/input -name run_q3.sh -path '*tools*' | head -1)")")
echo "bundle found at: $SRC"
rm -rf /kaggle/working/ire-team && cp -r "$SRC" /kaggle/working/ire-team
cd /kaggle/working/ire-team && chmod +x tools/run_q3.sh
nvidia-smi -L
python -c "import torch, numpy, pandas, pyarrow; print('torch', torch.__version__, '| cuda GPUs', torch.cuda.device_count(), '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)"
ls reports/q3/

In [ ]:
%%bash
# 2. Smoke test: 50 batches of the full model on EB-NeRD. Must end with a test AUC line.
# If this fails with a pandas/pyarrow error, run:  pip install -q pandas==3.0.5 pyarrow==25.0.1  and retry.
set -e
cd /kaggle/working/ire-team
python -u src/baseline/train_nrms.py --config config/ebnerd.yaml --popularity --freshness \
    --epochs 1 --max-steps 50 --val-sample 1000 --test-sample 1000 --tag smoke --n-boot 50 2>&1 | grep -v -i warn
rm -f reports/q3/*smoke* data/feature_store/*/q3_runs/smoke*

In [ ]:
%%bash
# 3. The queue. Two GPUs: EB-NeRD on one, MIND on the other, in parallel. One GPU: sequential.
# Runs whose reports/q3/*.json already exist are skipped.
cd /kaggle/working/ire-team
export PYTHON=python
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  CUDA_VISIBLE_DEVICES=0 tools/run_q3.sh ebnerd > worker_ebnerd.txt 2>&1 &
  CUDA_VISIBLE_DEVICES=1 tools/run_q3.sh mind   > worker_mind.txt   2>&1 &
  wait
else
  tools/run_q3.sh > worker_all.txt 2>&1
fi
cat worker_*.txt

In [ ]:
%%bash
# 4. Package results (reports, per-candidate test scores, logs; no .pt checkpoints) for download.
cd /kaggle/working/ire-team
zip -qr /kaggle/working/q3_results.zip reports/q3 data/feature_store/*/q3_runs logs worker_*.txt -x '*.pt'
ls -lh /kaggle/working/q3_results.zip
unzip -l /kaggle/working/q3_results.zip | grep json
rm -rf /kaggle/working/ire-team   # keep the saved output small